# Zero-Shot Learning for Spam Detection

**Zero-shot learning** is a setup in which a model classifies examples into categories it was never explicitly trained on. A large language model is pretrained on a great deal of text, and at inference time it can be asked to sort new text into labels it has never seen during training. The same idea applies to text, images, and other domains.

In notebook 1 we trained a classifier on labelled spam data. Here we take the opposite approach: we use a pretrained Transformer with no training of our own, and ask it to label messages as spam or not.

## Learning Objectives

At the end of this notebook, you should be able to:

- Build a zero-shot classification pipeline with the Transformers library.
- Apply a pretrained model to label messages without any task-specific training.
- Evaluate the predictions with the Hamming loss and compare them to a trained classifier.

## Set up libraries and data

In [ ]:
from transformers import pipeline
import pandas as pd
from tqdm import tqdm

In [ ]:
data = pd.read_csv(
    "../data/SMSSpamCollection.txt",
    encoding="utf-8",
    header=None,
    delimiter="\t",
    names=["target", "text"],
)

In [ ]:
data.head(5)

In [ ]:
print(f"There are {data.shape[0]} rows in the dataset")

## Zero-shot spam classification

The `pipeline` function from the Transformers library wraps model download, tokenisation, and inference into a single object. For zero-shot classification it uses a model trained on natural language inference, which lets us pass any set of candidate labels at inference time. We give it the message text and our candidate labels, and it returns a score for each label.

### Preparing the pipeline in one-line of code

In [ ]:
classifier = pipeline(
    "zero-shot-classification", device=0
)  # device=0 uses the first GPU; set device=-1 to run on CPU

By default the zero-shot pipeline loads [**facebook/bart-large-mnli**](https://huggingface.co/facebook/bart-large-mnli), a BART model fine-tuned on the MultiNLI natural language inference dataset. Because it was trained to judge whether one sentence entails another, it can score any candidate label we provide as a hypothesis, which is what makes zero-shot classification work.

To learn more, see the [model card](https://huggingface.co/facebook/bart-large-mnli) and the [BART paper](https://arxiv.org/pdf/1910.13461).

### Making Predictions

The model works best with **informative labels**. The raw tags `spam` and `ham` carry little meaning for a general-purpose model, so we map them to more descriptive phrases: `click bait` and `written by humans`. The exact Hamming loss depends on the model version, but more descriptive labels generally give the model a better chance.

Can you find label descriptions that lower the Hamming loss further?

In [ ]:
category_map = {"spam": "click bait", "ham": "written by humans"}

In [ ]:
candidate_labels = list(category_map.values())
predictedCategories = []
trueCategories = []
for i in tqdm(range(100)):
    text = data.iloc[i,]["text"]
    cat = [data.iloc[i,]["target"]]
    res = classifier(text, candidate_labels, multi_label=False)
    labels = res["labels"]
    scores = res["scores"]  # extracting the scores associated with the labels
    res_dict = {label: score for label, score in zip(labels, scores)}
    sorted_dict = dict(
        sorted(res_dict.items(), key=lambda x: x[1], reverse=True)
    )  # sorting the dictionary of labels in descending order based on their score
    categories = next(k for i, (k, v) in enumerate(sorted_dict.items()))

    predictedCategories.append(categories)
    trueCats = [category_map[x] for x in cat]
    trueCategories.append(trueCats)

In [ ]:
for y_true, y_pred in zip(trueCategories[:3], predictedCategories[:3]):
    print(f"True Categories {y_true}")
    print(f"Predicted Categories {y_pred}")
    print("#" * 50)

### Hamming Loss
The Hamming loss is the fraction of labels that are incorrectly predicted.

In [ ]:
from sklearn.metrics import hamming_loss

print(
    f"The hamming loss is {hamming_loss(trueCategories, predictedCategories):.4f} compared to 0.0237 from the last trained model in Notebook 1"
)

**Result:** with the labels `click bait` and `written by humans`, the Hamming loss on the first 100 messages is about 0.35, so roughly a third of the messages are misclassified. That is much worse than the trained classifier in notebook 1, whose Hamming loss was around 0.02.

The trade-off is the point of this notebook. Zero-shot classification needs no labelled data and no training step, which makes it useful for quick prototyping or when you have no labels at all. A small model trained directly on the task still wins on accuracy, so zero-shot is a convenient starting point, not a free replacement for a trained classifier.

## Check your understanding

Can you find better "labels" for the category description improve the Hamming Loss?

### Your turn: engineer better labels

The descriptive labels lowered the Hamming loss, but you can probably do better. Compare a few candidate-label sets and measure which one wins.

**Steps**
1. Define 2 or 3 label maps, for example `{"spam": "advertising", "ham": "personal message"}`.
2. For each map, classify the first 100 messages (reuse the loop above) and compute the Hamming loss.
3. Print the Hamming loss for each label set and note which is best.

**Hint:** wrap the prediction loop from earlier in a function `evaluate_labels(category_map)` that returns the Hamming loss, then call it for each candidate set.

**Expected:** the Hamming losses differ between label sets; more concrete, descriptive labels usually score lower (better).

In [ ]:
from sklearn.metrics import hamming_loss


def evaluate_labels(category_map):
    labels = list(category_map.values())
    predicted, true = [], []
    # Loop over the first 100 messages, classify each, and collect labels
    for i in range(100):
        text = data.iloc[i]["text"]
        res = classifier(text, labels, multi_label=False)
        predicted.append(res["labels"][0])
        true.append([category_map[data.iloc[i]["target"]]])
    # Return the Hamming loss between true and predicted
    return hamming_loss(true, predicted)


# Try a few label maps and print the Hamming loss for each
label_sets = {
    "clickbait/human": {"spam": "click bait", "ham": "written by humans"},
    "advertising/personal": {"spam": "advertising", "ham": "personal message"},
    "promotion/conversation": {
        "spam": "promotional offer",
        "ham": "personal conversation",
    },
}
for name, cmap in label_sets.items():
    print(name, "->", evaluate_labels(cmap))

**Explanation**

Wrapping the loop in a function makes it easy to compare label sets on the same 100 messages. The Hamming loss changes with the wording: concrete, descriptive labels usually beat vague ones, because the NLI model scores them as more plausible hypotheses. This is label engineering, the zero-shot equivalent of prompt engineering.


## Summary

In this notebook you:

- Built a zero-shot classification pipeline with a pretrained Transformer.
- Applied it to label SMS messages with no task-specific training.
- Evaluated the predictions with the Hamming loss and compared them to the trained classifier from notebook 1.

Zero-shot classification is convenient when labelled data is scarce, but a model trained on the task gives better accuracy here.

## References & Further Reading

- [**Hugging Face: Zero-Shot Classification**](https://huggingface.co/tasks/zero-shot-classification): What zero-shot classification is and which models suit it.
- [**Hugging Face: Pipelines**](https://huggingface.co/docs/transformers/main_classes/pipelines): Reference for the `pipeline` API used here.